In [1]:
!pip install transformers datasets torch scikit-learn nltk -q

In [154]:

import pandas as pd
import numpy as np
import re
import string
import nltk
import torch
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [155]:


from google.colab import files
uploaded = files.upload()

df = pd.read_csv("customer_support_tickets.csv")

print(df.head())
print(df.shape)
print(df.info())

Saving customer_support_tickets.csv to customer_support_tickets (2).csv
   Ticket ID        Customer Name              Customer Email  Customer Age  \
0          1        Marisa Obrien  carrollallison@example.com            32   
1          2         Jessica Rios    clarkeashley@example.com            42   
2          3  Christopher Robbins   gonzalestracy@example.com            48   
3          4     Christina Dillon    bradleyolson@example.org            27   
4          5    Alexander Carroll     bradleymark@example.com            67   

  Customer Gender Product Purchased Date of Purchase      Ticket Type  \
0           Other        GoPro Hero       2021-03-22  Technical issue   
1          Female       LG Smart TV       2021-05-22  Technical issue   
2           Other          Dell XPS       2020-07-14  Technical issue   
3          Female  Microsoft Office       2020-11-13  Billing inquiry   
4          Female  Autodesk AutoCAD       2020-02-04  Billing inquiry   

             T

In [156]:
drop_cols = ['Ticket ID', 'Customer Name', 'Customer Email','Customer Age','Customer Gender','Date of Purchase','Ticket Channel','Fist Response Time','Product Purchased'
,'Ticket Status','Resolution','First Response Time','Customer Satisfaction Rating']
df.drop(columns=drop_cols, inplace=True, errors='ignore')

print(df.columns)

Index(['Ticket Type', 'Ticket Subject', 'Ticket Description',
       'Ticket Priority', 'Time to Resolution'],
      dtype='object')


In [157]:
df.isnull().sum()

,0
Ticket Type,0
Ticket Subject,0
Ticket Description,0
Ticket Priority,0
Time to Resolution,5700


In [159]:
import pandas as pd
import numpy as np


time_series = pd.to_datetime(df["Time to Resolution"], format="%H:%M", errors="coerce")

minutes = time_series.dt.hour * 60 + time_series.dt.minute

minutes_filled = minutes.fillna(minutes.mean())

# Convert back to HH:MM format
df["hours_min"] = (
    pd.to_datetime(minutes_filled, unit="m")
    .dt.strftime("%H:%M")
)

In [160]:
df['Ticket Subject'] = df['Ticket Subject'].fillna('')
df['Ticket Description'] = df['Ticket Description'].fillna('')

df = df.dropna(subset=['Ticket Type'])

In [161]:
df['bert_input'] = (
    df['Ticket Subject'].fillna('') + " " +
    df['Ticket Description'].fillna('')
)

In [162]:
import re

def bert_clean(text):

    text = str(text).lower()

    # remove urls
    text = re.sub(r"http\S+", " ", text)

    # remove html
    text = re.sub(r"<.*?>", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [163]:
df['bert_input'] = df['bert_input'].apply(
    bert_clean
)

In [77]:
#text cleaning
//////
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()


    text = re.sub(r'<.*?>', '', text)


    text = re.sub(r'http\S+|www\S+', '', text)


    text = text.translate(str.maketrans('', '', string.punctuation))


    text = re.sub(r'\d+', '', text)


    words = text.split()


    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

In [78]:
///df['clean_text'] = df['text'].apply(clean_text)

print(df[['text', 'clean_text']].head())

                                                text  \
0  Product setup I'm having an issue with the {pr...   
1  Peripheral compatibility I'm having an issue w...   
2  Network problem I'm facing a problem with my {...   
3  Account access I'm having an issue with the {p...   
4  Data loss I'm having an issue with the {produc...   

                                          clean_text  
0  product setup im issue productpurchased please...  
1  peripheral compatibility im issue productpurch...  
2  network problem im facing problem productpurch...  
3  account access im issue productpurchased pleas...  
4  data loss im issue productpurchased please ass...  


In [79]:
///df.drop_duplicates(subset=['clean_text'], inplace=True)

print("Dataset shape after duplicates removal:", df.shape)

Dataset shape after duplicates removal: (8274, 7)


In [26]:
df['Ticket Priority'].head()

,Ticket Priority
0,Critical
1,Critical
2,Low
3,Low
4,Low


In [174]:
# =========================================================
# PRIORITY ENCODER
# =========================================================

from sklearn.preprocessing import LabelEncoder
import pickle


priority_encoder = LabelEncoder()

df['priority_label'] = priority_encoder.fit_transform(
    df['Ticket Priority']
)


with open("priority_encoder.pkl", "wb") as f:

    pickle.dump(priority_encoder, f)


print(
    dict(
        zip(
            priority_encoder.classes_,
            priority_encoder.transform(
                priority_encoder.classes_
            )
        )
    )
)

{'Critical': np.int64(0), 'High': np.int64(1), 'Low': np.int64(2), 'Medium': np.int64(3)}


In [164]:
from sklearn.preprocessing import LabelEncoder
import pickle

label_encoder = LabelEncoder()

df['ticket_type_label'] = label_encoder.fit_transform(
    df['Ticket Type']
)

In [166]:
from sklearn.preprocessing import LabelEncoder
import pickle



ticket_type_encoder = LabelEncoder()



df['ticket_type_label'] = ticket_type_encoder.fit_transform(
    df['Ticket Type']
)



with open("ticket_type_encoder.pkl", "wb") as f:

    pickle.dump(ticket_type_encoder, f)



print(
    dict(
        zip(
            ticket_type_encoder.classes_,
            ticket_type_encoder.transform(
                ticket_type_encoder.classes_
            )
        )
    )
)

{'Billing inquiry': np.int64(0), 'Cancellation request': np.int64(1), 'Product inquiry': np.int64(2), 'Refund request': np.int64(3), 'Technical issue': np.int64(4)}


In [167]:
df['Time to Resolution'] = pd.to_datetime(
    df['Time to Resolution'],
    errors='coerce'
)

In [168]:
df['hours_min'] = (
    df['Time to Resolution'].dt.hour +
    df['Time to Resolution'].dt.minute / 60
)
df['hours_min'] = df['hours_min'].fillna(
    df['hours_min'].median()
)
print(
    df[
        ['Time to Resolution', 'hours_min']
    ].head()
)

   Time to Resolution  hours_min
0                 NaT  11.816667
1                 NaT  11.816667
2 2023-06-01 18:05:38  18.083333
3 2023-06-01 01:57:40   1.950000
4 2023-06-01 19:53:42  19.883333


In [169]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df['hours_min'] = scaler.fit_transform(
    df[['hours_min']]
)

In [170]:
X = df['bert_input']

In [171]:
y_type = df['ticket_type_label']

In [175]:
y_priority = df['priority_label']

In [176]:
y_time = df['hours_min']

In [61]:
!pip install transformers accelerate -q

In [177]:
import torch
import torch.nn as nn
import numpy as np

from torch.utils.data import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error
)

In [178]:
X_train, X_test, \
y_type_train, y_type_test, \
y_priority_train, y_priority_test, \
y_time_train, y_time_test = train_test_split(

    X,
    y_type,
    y_priority,
    y_time,

    test_size=0.2,

    random_state=42,

    stratify=y_type
)

In [179]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [180]:
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=64
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=64
)

In [181]:


class MultiTaskDataset(Dataset):

    def __init__(
        self,
        encodings,
        y_type,
        y_priority,
        y_time
    ):

        self.encodings = encodings

        self.y_type = list(y_type)

        self.y_priority = list(y_priority)

        self.y_time = list(y_time)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels_type'] = torch.tensor(
            self.y_type[idx],
            dtype=torch.long
        )

        item['labels_priority'] = torch.tensor(
            self.y_priority[idx],
            dtype=torch.long
        )

        item['labels_time'] = torch.tensor(
            self.y_time[idx],
            dtype=torch.float
        )

        return item

    def __len__(self):

        return len(self.y_type)

In [182]:
train_dataset = MultiTaskDataset(
    train_encodings,
    y_type_train,
    y_priority_train,
    y_time_train
)

test_dataset = MultiTaskDataset(
    test_encodings,
    y_type_test,
    y_priority_test,
    y_time_test
)

In [183]:


class MultiTaskBERT(nn.Module):

    def __init__(
        self,
        num_ticket_labels,
        num_priority_labels
    ):

        super(MultiTaskBERT, self).__init__()

        self.bert = DistilBertModel.from_pretrained(
            "distilbert-base-uncased"
        )

        hidden_size = 768

        # Ticket Type Head
        self.ticket_classifier = nn.Linear(
            hidden_size,
            num_ticket_labels
        )

        # Priority Head
        self.priority_classifier = nn.Linear(
            hidden_size,
            num_priority_labels
        )

        # Regression Head
        self.time_regressor = nn.Linear(
            hidden_size,
            1
        )

        self.dropout = nn.Dropout(0.3)

    def forward(
        self,
        input_ids,
        attention_mask,
        labels_type=None,
        labels_priority=None,
        labels_time=None
    ):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0]

        pooled_output = self.dropout(
            pooled_output
        )

        # Outputs
        ticket_logits = self.ticket_classifier(
            pooled_output
        )

        priority_logits = self.priority_classifier(
            pooled_output
        )

        time_output = self.time_regressor(
            pooled_output
        ).squeeze()

        loss = None

        if labels_type is not None:

            loss_fct_class = nn.CrossEntropyLoss()

            loss_fct_reg = nn.MSELoss()

            loss_ticket = loss_fct_class(
                ticket_logits,
                labels_type
            )

            loss_priority = loss_fct_class(
                priority_logits,
                labels_priority
            )

            loss_time = loss_fct_reg(
                time_output,
                labels_time
            )

            # Total combined loss
            loss = (
                  loss_ticket +
                  loss_priority +
                  (0.01 * loss_time)
                    )

        return {
            "loss": loss,
            "ticket_logits": ticket_logits,
            "priority_logits": priority_logits,
            "time_output": time_output
        }

In [184]:
#model init
num_ticket_labels = len(
    df['ticket_type_label'].unique()
)

num_priority_labels = len(
    df['priority_label'].unique()
)

model = MultiTaskBERT(
    num_ticket_labels,
    num_priority_labels
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [185]:


training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    learning_rate=3e-5,

    weight_decay=0.01,

    logging_steps=50,

    report_to="none"
)

In [186]:
#train
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

In [187]:
trainer.train()

Step,Training Loss
50,3.043750
100,3.023346
150,3.024940
200,3.018398
250,3.011752
300,3.014473
350,3.015961
400,3.027302
450,3.010043
500,3.002952


TrainOutput(global_step=4235, training_loss=2.923422733023147, metrics={'train_runtime': 460.0347, 'train_samples_per_second': 73.636, 'train_steps_per_second': 9.206, 'total_flos': 0.0, 'train_loss': 2.923422733023147, 'epoch': 5.0})

In [138]:
predictions = trainer.predict(test_dataset)

In [139]:
predictions

PredictionOutput(predictions=(array([[-0.03579792, -0.12421234, -0.51677585,  0.72040826,  0.2959529 ],
       [ 0.4088384 ,  0.5904972 ,  0.2343528 , -0.39088216, -0.06783164],
       [ 0.18257035, -0.34394586, -0.2964219 ,  0.3130231 , -0.20010115],
       ...,
       [-0.02509002, -0.33621207,  0.03249119, -0.06979589,  0.52392143],
       [-0.11689007, -0.03888028,  0.4220001 ,  0.17459324, -0.09597556],
       [-0.3354268 ,  0.12536465,  0.2305762 ,  0.3542252 , -0.01787062]],
      dtype=float32), array([[-0.36691093, -0.69675255,  0.31643686,  0.5118268 ],
       [-0.5503971 ,  0.5063817 ,  0.34119067, -0.64726555],
       [-0.5312009 ,  0.20593043,  0.13025574,  0.7275429 ],
       ...,
       [-0.12532243,  0.27304414, -1.0449681 ,  0.13400812],
       [ 0.23216191, -0.4145385 , -0.79564506,  1.1206884 ],
       [ 0.30846044, -0.13616037,  0.1329151 , -0.20399335]],
      dtype=float32), array([0.3582606 , 0.7735333 , 0.4787102 , ..., 0.31182662, 0.45249456,
       0.45606512]

In [188]:
torch.save(
    model.state_dict(),
    "multitask_bert_model.pth"
)

In [189]:
tokenizer.save_pretrained(
    "multitask_tokenizer"
)

('multitask_tokenizer/tokenizer_config.json',
 'multitask_tokenizer/tokenizer.json')

In [190]:
//import pickle

metadata = {
    "num_ticket_labels": len(df['ticket_type_label'].unique()),
    "num_priority_labels": len(df['priority_label'].unique())
}

with open("model_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

In [194]:
metadata = {

    "num_ticket_labels":
        len(ticket_type_encoder.classes_),

    "num_priority_labels":
        len(priority_encoder.classes_)
}

with open("model_metadata.pkl", "wb") as f:

    pickle.dump(metadata, f)

print("Metadata Saved")

Metadata Saved


In [191]:
import pickle

with open("ticket_type_encoder.pkl", "wb") as f:

    pickle.dump(ticket_type_encoder, f)

print("Ticket Type Encoder Saved")

Ticket Type Encoder Saved


In [192]:
with open("priority_encoder.pkl", "wb") as f:

    pickle.dump(priority_encoder, f)

print("Priority Encoder Saved")

Priority Encoder Saved


In [193]:
with open("time_scaler.pkl", "wb") as f:

    pickle.dump(scaler, f)

print("Scaler Saved")

Scaler Saved


In [195]:
!zip -r multitask_tokenizer.zip multitask_tokenizer

  adding: multitask_tokenizer/ (stored 0%)
  adding: multitask_tokenizer/tokenizer_config.json (deflated 42%)
  adding: multitask_tokenizer/tokenizer.json (deflated 71%)


In [196]:
from google.colab import files

files.download("multitask_bert_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [197]:
files.download("multitask_tokenizer.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [198]:
files.download("ticket_type_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [199]:
files.download("priority_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [200]:
files.download("time_scaler.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [201]:
files.download("model_metadata.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [148]:
#reversing the priority
priority_mapping = {
    0: "Critical",
    1: "High",
    2: "Medium",
    3: "Low"
}

NameError: name 'inputs' is not defined

In [150]:
predicted_priority_id = torch.argmax(
    outputs['priority_logits'],
    dim=1
).item()

NameError: name 'outputs' is not defined

In [149]:
predicted_priority = priority_mapping[
    predicted_priority_id
]

print(predicted_priority)

NameError: name 'predicted_priority_id' is not defined

In [ ]:
#savemodel


////

torch.save(
    model.state_dict(),
    "multitask_bert_model.pth"
)

tokenizer.save_pretrained(
    "multitask_tokenizer"
)

print("Multi-task BERT Model Saved")

In [ ]:
no need to run below one

In [52]:
df['ticket_type_label'] = pd.to_numeric(
    df['ticket_type_label'],
    errors='coerce'
)

df['priority_label'] = pd.to_numeric(
    df['priority_label'],
    errors='coerce'
)

df['hours_min'] = pd.to_numeric(
    df['hours_min'],
    errors='coerce'
)

In [53]:
df = df.dropna(
    subset=[
        'ticket_type_label',
        'priority_label',
        'hours_min',
        'clean_text'
    ]
)

In [54]:
df['ticket_type_label'] = df[
    'ticket_type_label'
].astype(int)

df['priority_label'] = df[
    'priority_label'
].astype(int)

df['hours_min'] = df[
    'hours_min'
].astype(float)

In [55]:
print(df[['ticket_type_label',
          'priority_label',
          'hours_min']].dtypes)

ticket_type_label      int64
priority_label         int64
hours_min            float64
dtype: object


In [67]:
print(df.shape)

print(df[['ticket_type_label',
          'priority_label',
          'hours_min',
          'clean_text']].isna().sum())

(0, 11)
ticket_type_label    0
priority_label       0
hours_min            0
clean_text           0
dtype: int64


In [70]:
df['hours_min'].head(10)

,hours_min


In [ ]:
accuracy


In [202]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

In [203]:
predictions = trainer.predict(test_dataset)

In [204]:
ticket_logits = predictions.predictions[0]

priority_logits = predictions.predictions[1]

time_predictions = predictions.predictions[2]

In [205]:
ticket_preds = np.argmax(
    ticket_logits,
    axis=1
)

priority_preds = np.argmax(
    priority_logits,
    axis=1
)

In [206]:
true_ticket = np.array(y_type_test)

true_priority = np.array(y_priority_test)

true_time = np.array(y_time_test)

In [207]:
ticket_accuracy = accuracy_score(
    true_ticket,
    ticket_preds
)

ticket_f1 = f1_score(
    true_ticket,
    ticket_preds,
    average='weighted'
)

print("===================================")

print("TICKET TYPE RESULTS")

print("===================================")

print("Accuracy:", ticket_accuracy)

print("F1 Score:", ticket_f1)

print("\nClassification Report:\n")

print(
    classification_report(
        true_ticket,
        ticket_preds
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        true_ticket,
        ticket_preds
    )
)

TICKET TYPE RESULTS
Accuracy: 0.19893742621015348
F1 Score: 0.1959572560851009

Classification Report:

              precision    recall  f1-score   support

           0       0.16      0.15      0.16       327
           1       0.18      0.13      0.15       339
           2       0.19      0.26      0.22       328
           3       0.23      0.26      0.24       351
           4       0.22      0.19      0.20       349

    accuracy                           0.20      1694
   macro avg       0.20      0.20      0.20      1694
weighted avg       0.20      0.20      0.20      1694


Confusion Matrix:

[[50 47 91 77 62]
 [71 43 98 69 58]
 [57 43 86 79 63]
 [69 45 88 91 58]
 [68 55 83 76 67]]


In [208]:
priority_accuracy = accuracy_score(
    true_priority,
    priority_preds
)

priority_f1 = f1_score(
    true_priority,
    priority_preds,
    average='weighted'
)

print("===================================")

print("PRIORITY RESULTS")

print("===================================")

print("Accuracy:", priority_accuracy)

print("F1 Score:", priority_f1)

print("\nClassification Report:\n")

print(
    classification_report(
        true_priority,
        priority_preds
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        true_priority,
        priority_preds
    )
)

PRIORITY RESULTS
Accuracy: 0.24439197166469895
F1 Score: 0.2442464493226308

Classification Report:

              precision    recall  f1-score   support

           0       0.25      0.25      0.25       441
           1       0.26      0.25      0.25       424
           2       0.23      0.27      0.24       375
           3       0.24      0.22      0.23       454

    accuracy                           0.24      1694
   macro avg       0.24      0.25      0.24      1694
weighted avg       0.25      0.24      0.24      1694


Confusion Matrix:

[[110  95 115 121]
 [107 105 102 110]
 [103  92 100  80]
 [112 117 126  99]]


In [209]:
mae = mean_absolute_error(
    true_time,
    time_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_time,
        time_predictions
    )
)

r2 = r2_score(
    true_time,
    time_predictions
)

print("===================================")

print("RESOLUTION TIME RESULTS")

print("===================================")

print("MAE:", mae)

print("RMSE:", rmse)

print("R2 Score:", r2)

RESOLUTION TIME RESULTS
MAE: 0.1916889533824617
RMSE: 0.2469313248593666
R2 Score: -1.1412184161168577
